# Step 04.4 — Build the First MLP

现在把前面三节真正拼起来：

```text
Linear -> GELU -> Linear
```

这就是 Transformer 中经典 feed-forward network / MLP 的核心形式。

本节目标：

1. 理解 `C -> 4C -> C`；
2. 自己写一个 MLX `MLP(nn.Module)`；
3. 追踪 `(B,T,C)` 的 shape；
4. 证明 MLP 不会混合不同 token；
5. 算清楚参数量。

## 1. 为什么先升维：`C -> 4C -> C`

假设每个 token 当前有 `C=4` 个特征：

```text
x = [x0, x1, x2, x3]
```

如果直接做：

```text
4 -> 4
```

当然也能做非线性变换，但中间可用的 feature channels 比较少。

经典 Transformer / GPT 风格 MLP 常先扩展：

```text
4 -> 16 -> GELU -> 4
```

也就是一般形式：

```text
C -> 4C -> GELU -> C
```

可以把中间的 `4C` 暂时理解成一个更大的**特征工作区**：第一层产生更多不同的加权组合，GELU 对这些中间特征做非线性筛选，第二层再把它们重新组合回 `C` 维。

注意：`4C` **不是数学定律**。它是经典 Transformer/GPT 中常见的设计比例；现代模型也会使用别的 expansion ratio 或 SwiGLU 等结构。

## 2. 用 MLX 写出第一个 MLP

我们先用很小的 `C=4`，这样所有 shape 都容易看。

In [ ]:
import mlx.core as mx
import mlx.nn as nn


class MLP(nn.Module):
    def __init__(self, n_embd: int):
        super().__init__()

        self.c_fc = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd, bias=False)

    def __call__(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x


mlp = MLP(n_embd=4)
print(mlp)

逐行看结构：

```python
self.c_fc = nn.Linear(4, 16, bias=False)
```

第一层把最后一个 feature axis：

```text
4 -> 16
```

然后：

```python
self.gelu = nn.GELU()
```

GELU 对 16 个中间值逐元素做非线性变换，shape 不变。

最后：

```python
self.c_proj = nn.Linear(16, 4, bias=False)
```

再把 16 个中间特征混合回 4 个输出特征。

### 把 forward 拆开看 shape

不要只执行 `mlp(x)`，先把内部三步单独运行一次。

In [ ]:
x = mx.array([[1.0, 2.0, 3.0, 4.0]])

h1 = mlp.c_fc(x)
h2 = mlp.gelu(h1)
y = mlp.c_proj(h2)

mx.eval(h1, h2, y)

print("input :", x.shape)
print("c_fc  :", h1.shape)
print("GELU  :", h2.shape)
print("c_proj:", y.shape)

应该得到：

```text
(1,4)
  ↓ c_fc
(1,16)
  ↓ GELU
(1,16)
  ↓ c_proj
(1,4)
```

注意：最终维度回到 4，不代表“恢复了原来的 4 个数”。

输出已经是经过两次 feature mixing 和一次非线性变换后的**新表示**。

## 3. 真正 MiniGPT 的输入是 `(B,T,C)`

现在构造：

```text
B = 2
T = 3
C = 4
```

也就是：2 个样本，每个样本 3 个 token，每个 token 有 4 个特征。

In [ ]:
X = mx.arange(2 * 3 * 4, dtype=mx.float32).reshape(2, 3, 4)

H1 = mlp.c_fc(X)
H2 = mlp.gelu(H1)
Y = mlp.c_proj(H2)

mx.eval(H1, H2, Y)

print("X  :", X.shape)
print("H1 :", H1.shape)
print("H2 :", H2.shape)
print("Y  :", Y.shape)

shape 流转：

```text
(B,T,C) = (2,3,4)
        ↓ Linear(4,16)
(2,3,16)
        ↓ GELU
(2,3,16)
        ↓ Linear(16,4)
(2,3,4)
```

这里最重要的是：

> Linear 只改变最后一个 feature axis；`B` 和 `T` 都没有被混到一起。

## 4. MLP 会不会让不同 token 互相交流？

不会。

对 `(B,T,C)` 来说，MLP 可以想成：

```text
token 0: [C个特征] -> 同一个 MLP -> [C个新特征]
token 1: [C个特征] -> 同一个 MLP -> [C个新特征]
token 2: [C个特征] -> 同一个 MLP -> [C个新特征]
```

所有 token **共享同一套 MLP 参数**，但一个 token 的输入不会进入另一个 token 的计算。

我们可以直接做实验验证。

In [ ]:
# 一个 batch，两个 token，每个 token 4 个特征
A = mx.array([[[1.0, 2.0, 3.0, 4.0],
               [5.0, 6.0, 7.0, 8.0]]])

# 只修改第二个 token，第一个 token 完全不动
B = mx.array([[[1.0, 2.0, 3.0, 4.0],
               [50.0, 60.0, 70.0, 80.0]]])

YA = mlp(A)
YB = mlp(B)
mx.eval(YA, YB)

print("first token unchanged:", bool(mx.allclose(YA[:, 0, :], YB[:, 0, :])))
print("second token changed  :", not bool(mx.allclose(YA[:, 1, :], YB[:, 1, :])))

如果看到：

```text
first token unchanged: True
second token changed  : True
```

就说明：改变 token 1 不会影响 token 0 的 MLP 输出。

所以以后 Transformer 中可以先形成一个非常重要的分工直觉：

```text
Attention -> 负责不同 token / 位置之间的信息交流
MLP       -> 负责每个 token 内部的 feature transformation
```

Attention 还没有正式学，这里只先记住这个分工。

## 5. 参数量为什么大约是 `8C²`？

我们这里使用 `bias=False`。

第一层：

```text
Linear(C, 4C)
weight.shape = (4C, C)
参数量 = 4C²
```

第二层：

```text
Linear(4C, C)
weight.shape = (C, 4C)
参数量 = 4C²
```

总共：

```text
4C² + 4C² = 8C²
```

GELU 没有自己的可学习参数。

In [ ]:
from mlx.utils import tree_flatten

flat = tree_flatten(mlp.parameters())

for name, value in flat:
    print(name, value.shape, "count =", value.size)

total = sum(value.size for _, value in flat)
print("total parameters:", total)
print("formula 8*C^2    :", 8 * 4 * 4)

对于当前教学例子 `C=4`：

```text
8 × 4² = 128 parameters
```

以后 MiniGPT 使用 `C=128` 时，仅一个这种 bias-free MLP 就有：

```text
8 × 128² = 131,072 parameters
```

所以 MLP 并不是 Transformer 里的小配角，它本身就占据相当大的参数量和计算量。

## 最终写法

理解清楚以后，我们可以把 forward 压回一行：

```python
class MLP(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd, bias=False)

    def __call__(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))
```

现在这段代码不应该再是黑盒：

```text
x
↓ c_fc: feature expansion / mixing
4C
↓ GELU: non-linearity
4C
↓ c_proj: feature mixing / projection back
C
```

### Step 4 完成

你现在已经具备：

- Linear
- Parameter / bias
- `nn.Module`
- parameter tree
- ReLU / GELU
- MLP

下一步进入 Step 5：**神经网络到底怎样学习——prediction、loss、导数、gradient 和 gradient descent。**